# NB_PRC_Conform · Phase 2 · Bronze → LH_PRC_Silver
Types, deduplicates and decodes the Procurement EBS tables. Writes schema `silver`.

In [ ]:
%%configure -f
{"defaultLakehouse": {"name": "LH_PRC_Silver"}}

In [ ]:
BRONZE = "abfss://{{WS_PRC_BRONZE}}@onelake.dfs.fabric.microsoft.com/{{LH_PRC_BRONZE}}/Tables/ebs"
FX_TO_KWD = {"KWD": 1.0, "SAR": 0.0819, "AED": 0.0836}   # lab rates; EBS would use GL_DAILY_RATES

In [ ]:
from pyspark.sql import functions as F, Window as W
def bronze(t):
    df = spark.read.format("delta").load(f"{BRONZE}/{t}")
    df.createOrReplaceTempView(f"b_{t}")
    return df
for t in ["hr_operating_units", "mtl_categories_b", "mtl_system_items_b", "po_headers_all", "po_lines_all",
          "po_requisition_headers_all", "po_requisition_lines_all", "rcv_shipment_headers", "rcv_transactions"]:
    bronze(t)
spark.sql("SELECT * FROM VALUES " + ", ".join(f"('{k}', CAST({v} AS DOUBLE))" for k, v in FX_TO_KWD.items()) + " AS t(currency_code, rate_to_kwd)").createOrReplaceTempView("fx")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

def save(df, name, key):
    w = W.partitionBy(*key).orderBy(F.col("_load_ts").desc()) if "_load_ts" in df.columns else None
    if w is not None:
        df = df.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn", "_load_ts")
    df.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(f"silver.{name}")
    print(f"silver.{name}: {spark.table('silver.' + name).count()} rows")

In [ ]:
save(spark.sql("""
SELECT CAST(i.INVENTORY_ITEM_ID AS INT) item_id, i.SEGMENT1 item_code, i.DESCRIPTION item_name, i.PRIMARY_UOM_CODE uom,
       CAST(i.CATEGORY_ID AS INT) category_id, c.SEGMENT1 category_name, i._load_ts
FROM b_mtl_system_items_b i LEFT JOIN b_mtl_categories_b c ON c.CATEGORY_ID = i.CATEGORY_ID"""), "item", ["item_id"])

save(spark.sql("""
SELECT CAST(PO_HEADER_ID AS INT) po_header_id, SEGMENT1 po_number, CAST(ORG_ID AS INT) org_id, CAST(VENDOR_ID AS INT) supplier_id,
       CAST(VENDOR_SITE_ID AS INT) supplier_site_id, CURRENCY_CODE currency_code, CAST(CREATION_DATE AS DATE) order_date,
       INITCAP(AUTHORIZATION_STATUS) approval_status, CASE CLOSED_CODE WHEN 'OPEN' THEN 'Open' ELSE 'Closed' END closed_status,
       CAST(AGENT_ID AS INT) buyer_id, _load_ts
FROM b_po_headers_all"""), "po_header", ["po_header_id"])

save(spark.sql("""
SELECT CAST(l.PO_LINE_ID AS INT) po_line_id, CAST(l.PO_HEADER_ID AS INT) po_header_id, CAST(l.LINE_NUM AS INT) line_number,
       CAST(l.ORG_ID AS INT) org_id, CAST(l.ITEM_ID AS INT) item_id, CAST(l.CATEGORY_ID AS INT) category_id,
       CAST(l.QUANTITY AS DECIMAL(18,3)) quantity, l.UNIT_MEAS_LOOKUP_CODE uom, CAST(l.UNIT_PRICE AS DECIMAL(18,4)) unit_price,
       CAST(l.NEED_BY_DATE AS DATE) need_by_date, h.CURRENCY_CODE currency_code,
       CAST(l.QUANTITY * l.UNIT_PRICE AS DECIMAL(18,3)) line_amount,
       CAST(l.QUANTITY * l.UNIT_PRICE * fx.rate_to_kwd AS DECIMAL(18,3)) line_amount_kwd, l._load_ts
FROM b_po_lines_all l JOIN b_po_headers_all h ON h.PO_HEADER_ID = l.PO_HEADER_ID
JOIN fx ON fx.currency_code = h.CURRENCY_CODE"""), "po_line", ["po_line_id"])

save(spark.sql("""
SELECT CAST(t.TRANSACTION_ID AS INT) transaction_id, s.RECEIPT_NUM receipt_number, CAST(t.ORGANIZATION_ID AS INT) org_id,
       CAST(t.PO_HEADER_ID AS INT) po_header_id, CAST(t.PO_LINE_ID AS INT) po_line_id, INITCAP(t.TRANSACTION_TYPE) transaction_type,
       CAST(t.TRANSACTION_DATE AS DATE) transaction_date, CAST(t.QUANTITY AS DECIMAL(18,3)) quantity, t._load_ts
FROM b_rcv_transactions t JOIN b_rcv_shipment_headers s ON s.SHIPMENT_HEADER_ID = t.SHIPMENT_HEADER_ID"""), "receipt", ["transaction_id"])

save(spark.sql("""
SELECT CAST(l.REQUISITION_LINE_ID AS INT) requisition_line_id, h.SEGMENT1 requisition_number, CAST(h.ORG_ID AS INT) org_id,
       CAST(l.ITEM_ID AS INT) item_id, CAST(l.QUANTITY AS DECIMAL(18,3)) quantity, CAST(l.NEED_BY_DATE AS DATE) need_by_date,
       INITCAP(h.AUTHORIZATION_STATUS) status, l._load_ts
FROM b_po_requisition_lines_all l JOIN b_po_requisition_headers_all h ON h.REQUISITION_HEADER_ID = l.REQUISITION_HEADER_ID"""), "requisition", ["requisition_line_id"])

In [ ]:
# Data quality checks (DAT-02/03): fail the run if keys are missing or org_id is null
for t in ["po_header", "po_line", "receipt", "requisition"]:
    bad = spark.sql(f"SELECT COUNT(*) n FROM silver.{t} WHERE org_id IS NULL").first().n
    assert bad == 0, f"silver.{t} has {bad} rows without org_id"
print("Data quality checks passed")